In [1]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def query_local_llm():
    # 1. Initialize the local model via Ollama
    # By default, it connects to http://localhost:11434
    llm = ChatOllama(
        model="llama3.2",
        temperature=0.7,
    )
    
    # 2. Define a prompt template
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful and concise AI assistant."),
        ("user", "{input}")
    ])
    
    # 3. Create a chain using the LangChain Expression Language (LCEL)
    chain = prompt | llm | StrOutputParser()
    
    # 4. Invoke the chain
    user_question = "What are three fun facts about space?"
    print(f"Prompting local model: '{user_question}'\n")
    
    # Stream the response chunk by chunk as it generates
    for chunk in chain.stream({"input": user_question}):
        print(chunk, end="", flush=True)
    print()
 

/Users/zainabfirdaus/git/airflow/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
query_local_llm()

Prompting local model: 'What are three fun facts about space?'

Here are three fun facts about space:

1. **The Andromeda Galaxy is Coming for Us**: The Andromeda galaxy, our closest galactic neighbor, is currently approaching us at a speed of about 250,000 miles per hour (402,336 km/h). Don't worry, though - it won't collide with us for about 4 billion years!

2. **There's a Giant Storm on Jupiter that's been Raging for Centuries**: The Great Red Spot, a massive anticyclonic storm on Jupiter, has been continuously raging for at least 187 years and possibly much longer. It's so large that three Earths could fit inside it.

3. **The International Space Station Spins Like Crazy**: The ISS orbits the Earth at an incredible speed of about 17,500 miles per hour (28,200 km/h), which means it completes one rotation every 90 minutes. To make things more interesting, it also spins on its axis, resulting in a centrifugal force that's equivalent to a force of about 1/4th the strength of gravity.


In [1]:
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Setup the Sample Knowledge Base (Ingestion Data)
sample_raw_documents = [
    Document(page_content="Project Mercury was the first human spaceflight program of the United States, running from 1958 through 1963."),
    Document(page_content="The primary goal of Project Mercury was to orbit a manned spacecraft around Earth and investigate human capabilities in space."),
    Document(page_content="The internal code name for the project's tracking system was Project Vulcan, which was later changed to Mercury.")
]


/Users/zainabfirdaus/git/airflow/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

print("Initializing local embedding model...")
# 2. Initialize Ollama Embeddings for INGESTION
embeddings = OllamaEmbeddings(model="nomic-embed-text")


Initializing local embedding model...


In [3]:
print("Ingesting documents into local FAISS vector store...")
# 3. Process and store documents into a local vector database
vector_store = FAISS.from_documents(sample_raw_documents, embeddings)
retriever = vector_store.as_retriever(search_kwargs={"k": 2})

Ingesting documents into local FAISS vector store...


In [5]:
r=vector_store.asimilarity_search('Project Vulcan ')

In [ ]:
query = "What years did Project Mercury run?"

# Fetch the top 2 most relevant document chunks
docs = vector_store.similarity_search(query, k=2)

# Print the results neatly
for i, doc in enumerate(docs):
    print(f"--- Document {i+1} ---")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}\n")


In [6]:
for e in r :
    print(e.page_content)
    print(e.metadata)

TypeError: 'coroutine' object is not iterable

In [ ]:



print("Setting up Generative LLM...")
# 4. Initialize Ollama Chat LLM for GENERATION
llm = ChatOllama(model="llama3.2", temperature=0.2)

# 5. Define the RAG Prompt Template
rag_template = """
You are an expert historian. Answer the question based ONLY on the following context provided. 
If you do not know the answer, say that you don't know.

Context:
{context}

Question: {question}
Answer:"""

prompt = ChatPromptTemplate.from_template(rag_template)

# Helper function to format retrieved documents
def format_docs(docs):
    return "\n\n".join([doc.page_content for doc in docs])

# 6. Construct the RAG Chain using LangChain Expression Language (LCEL)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 7. Test the entire Local RAG setup
query = "What years did the first US human spaceflight program run?"
print(f"\nUser Query: {query}\n---")

for chunk in rag_chain.stream(query):
    print(chunk, end="", flush=True)
print("\n---")


/Users/zainabfirdaus/git/airflow/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Initializing local embedding model...
Ingesting documents into local FAISS vector store...
Setting up Generative LLM...

User Query: What years did the first US human spaceflight program run?
---
